# Day 1: Workflow Analysis

## Module 5: Automation in AI | Al Jazira Bank

In this notebook you will explore the AJB workflow inventory dataset, examine process characteristics, and begin identifying automation candidates based on data.

### What you will do
1. Load and inspect the workflow inventory dataset.
2. Explore distributions of manual hours, error rates, and complexity.
3. Identify patterns that distinguish strong automation candidates from poor ones.
4. Prepare observations to support your Lab A and Lab B deliverables.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the workflow inventory
workflows = pd.read_csv("../data/workflow_inventory.csv")

print(f"Dataset shape: {workflows.shape}")
print(f"Columns: {list(workflows.columns)}")
workflows.head()

In [ ]:
# Basic statistics for numeric columns
print("Summary statistics:")
print(workflows[["steps_count", "manual_hours_weekly", "error_rate_pct"]].describe())

print("\nAutomation potential distribution:")
print(workflows["automation_potential"].value_counts())

print("\nComplexity distribution:")
print(workflows["complexity"].value_counts())

print("\nDepartment distribution:")
print(workflows["department"].value_counts())

In [ ]:
# Exercise: Explore the relationship between manual hours and error rate
#
# Task 1: Create a scatter plot of manual_hours_weekly vs error_rate_pct.
# Task 2: Colour or label the points by automation_potential.
# Task 3: Which workflows have both high hours AND high error rates?
#          These are your strongest automation candidates.

fig, ax = plt.subplots(figsize=(10, 6))

colours = {"High": "#2ecc71", "Medium": "#f39c12", "Low": "#e74c3c"}
for potential, group in workflows.groupby("automation_potential"):
    ax.scatter(
        group["manual_hours_weekly"],
        group["error_rate_pct"],
        label=potential,
        c=colours.get(potential, "grey"),
        s=80,
        alpha=0.8,
    )
    for _, row in group.iterrows():
        ax.annotate(
            row["workflow_id"],
            (row["manual_hours_weekly"], row["error_rate_pct"]),
            fontsize=7,
            ha="left",
            va="bottom",
        )

ax.set_xlabel("Manual Hours per Week")
ax.set_ylabel("Error Rate (%)")
ax.set_title("Workflow Inventory: Manual Hours vs Error Rate")
ax.legend(title="Automation Potential")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise: Identify your top candidates
#
# Sort workflows by a composite of manual_hours_weekly and error_rate_pct.
# Filter for High or Medium automation potential.
# Write your observations below.

candidates = workflows[workflows["automation_potential"].isin(["High", "Medium"])].copy()
candidates["priority_score"] = (
    candidates["manual_hours_weekly"] / candidates["manual_hours_weekly"].max() * 0.5
    + candidates["error_rate_pct"] / candidates["error_rate_pct"].max() * 0.5
)
candidates = candidates.sort_values("priority_score", ascending=False)

print("Top candidates by composite score (hours + error rate):")
print(
    candidates[
        ["workflow_id", "process_name", "department", "manual_hours_weekly",
         "error_rate_pct", "automation_potential", "complexity", "priority_score"]
    ].head(10).to_string(index=False)
)

# Reflection:
# - Which workflows appear in your top 5?
# - Are any high-scoring workflows also high complexity? What does that mean for implementation?
# - Would you change the weighting between hours and error rate? Why?

## Exercise 4.5 - Copilot pressure test of your top automation candidate

Your composite priority score gives you a candidate. Now compare two GenAI models on the same question: would they actually recommend automating it inside an AJB context, and on what conditions?

You will use the SaintAGI training copilot's `compare()` to send the same prompt to two models and inspect the differences. Reading the disagreement is the lesson, not the agreement.

**Setup**

Ensure these env vars are set in the kernel that is running this notebook (see the Notebook Studio panel on your academy page):

```
SAINTAGI_BASE_URL=https://your-saintagi-host
SAINTAGI_INVITE_CODE=<your invite code>
SAINTAGI_PARTICIPANT_TOKEN=<your participant token>
SAINTAGI_MODULE_SLUG=automation-in-ai
```

In [ ]:
import pathlib
import sys

helper_dir = (pathlib.Path.cwd().resolve().parent.parent / "notebook-helpers")
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import saintagi_copilot as co

top_candidate = candidates.iloc[0]

prompt = (
    "You are advising AJB operations on automation feasibility. The proposed "
    "candidate is described below. Produce 6 bullets covering: 1) the single "
    "best automation pattern (RPA, workflow engine, ML model, LLM-assisted, or "
    "hybrid) and why, 2) the riskiest hidden dependency, 3) one regulator or "
    "audit angle (SAMA, SDAIA, PDPL) the team must respect, 4) a realistic "
    "first slice that could ship in 8 weeks, 5) a metric the head of operations "
    "would actually believe as proof of value, 6) one reason this might be the "
    "wrong workflow to automate first. Mark uncertainty explicitly. Do not "
    "invent figures.\n\n"
    "Candidate workflow:\n"
    f"- workflow_id: {top_candidate['workflow_id']}\n"
    f"- process_name: {top_candidate['process_name']}\n"
    f"- department: {top_candidate['department']}\n"
    f"- manual_hours_weekly: {top_candidate['manual_hours_weekly']}\n"
    f"- error_rate_pct: {top_candidate['error_rate_pct']}\n"
    f"- automation_potential: {top_candidate['automation_potential']}\n"
    f"- complexity: {top_candidate['complexity']}\n"
    f"- composite priority_score: {top_candidate['priority_score']:.2f}\n"
)

models_to_compare = [
    "openrouter/meta-llama/llama-3.3-70b-instruct:free",
    "openrouter/openai/gpt-4o-mini",
]

results = co.compare(
    prompt,
    models=models_to_compare,
    exercise_id="m5-d1-automation-compare",
    max_tokens=900,
)

for result in results:
    print("\n" + "=" * 80)
    print(f"MODEL: {result.model}")
    print(f"latency: {result.latency_ms} ms | tokens: {result.total_tokens} | cost: {result.cost_usd}")
    print("-" * 80)
    print(result.output if result.output else f"[no output] {result.error_message or ''}")

**Reflect on the comparison**

Capture in your workbook:

1. Where did the two models disagree? Was the disagreement substantive or stylistic?
2. Which model gave a recommendation you would actually walk into the head of operations meeting with? Why?
3. Did either model surface a regulator angle, hidden dependency, or "wrong workflow" warning that you had not considered? If yes, write a one-line update to your candidate's row in your shortlist.
4. Which model's view of "first slice in 8 weeks" feels more realistic for AJB? What would have to be true for that estimate to hold?